# simple usage

In [1]:
from multiprocessing import Pool
import h5py
import numpy as np

ic    = 0
imm   = 89 # 0-50 Mpc/h
fdir  = "/home/cossim/yuyu22/chenzhao/csst/simulation/"
fhalo = fdir + "c%04d/output/fullskyhalo-rockstar/mapsdir_%03d/halo_sub_%03d"%(ic, imm, imm)
with h5py.File(fhalo + ".0.hdf5", "r") as ff:
    print(ff.keys())
    ### Group   mean main halo 
    ### Subhalo mean subhalo
    print(ff['Group'].keys())
    print(ff['Subhalo'].keys())
    NumFiles = ff['Header'].attrs['NumFiles']
    print(NumFiles)
    print(ff['Header'].attrs['SnapNum'])
    # print(ff['Header'].attrs.keys())
    print(ff['Header'].attrs['Ngroups_Total'])
    print(ff['Header'].attrs['Nsubhalos_Total'])
def GetRADECZ(fname):
    with h5py.File(fname, "r") as ff:
        #### because some files do not contain data
        if "Group" in ff.keys():
            RA  = ff['Group/RA'][...]
            DEC = ff['Group/DEC'][...]
            Z   = ff['Group/Z'][...]
        else:
            RA  = np.zeros((0), dtype=np.float64)
            DEC = np.zeros((0), dtype=np.float64)
            Z   = np.zeros((0), dtype=np.float64)
    return RA, DEC, Z
fnames = [fhalo+".%d.hdf5"%(ii) for ii in range(NumFiles)]
out = np.array([GetRADECZ(fname) for fname in fnames], dtype=object)
RA  = np.concatenate(out[:,0])
DEC = np.concatenate(out[:,1])
Z   = np.concatenate(out[:,2])
print(RA.shape, DEC.shape, Z.shape)

<KeysViewHDF5 ['Group', 'Header', 'Subhalo']>
<KeysViewHDF5 ['DEC', 'GroupPos', 'ID', 'M200b', 'Mvir', 'PID', 'RA', 'Rs', 'Z']>
<KeysViewHDF5 ['DEC', 'ID', 'M200b', 'Mvir', 'PID', 'RA', 'Rs', 'SubhaloPos', 'Z']>
64
11
38186
5851
(38186,) (38186,) (38186,)


# use lightcone mask 

In [34]:
import sys
sys.path.append("/home/chenzhao/csst/simulation/scripts/")
from LightConeInfo import lc_dict, is_in_cone
sys.path.append("/home/chenzhao/worksoftware/csst-simulations-read/KunRead/")
from io_utils import IO_Halo_single, IO_Halo_multifile, get_available_fields
from glob import glob
import numpy as np
import camb
import h5py
from numpy.lib import recfunctions as rfn

def Getcamb(hdf5file):
    import h5py
    header = {}
    with h5py.File(hdf5file, 'r') as ff:
        for ikey in ff['Parameters'].attrs.keys():
            header[ikey] = ff['Parameters'].attrs[ikey]
    pars = camb.CAMBparams()
    param_in = {}
    ddir = '/home/chenzhao/csst/cosmologies'
    h0 = header['HubbleParam']
    Omegab = header['OmegaBaryon']
    Omegac = header['Omega0'] - Omegab
    omch2  = Omegac*h0*h0
    ombh2  = Omegab*h0*h0
    mnus   = [header['m_nu1'], header['m_nu2'], header['m_nu3']]
    Nncdm  = len(np.where(mnus != 0)[0])
    pars.set_cosmology(H0=h0*100, omch2=omch2, ombh2=ombh2, 
                       num_massive_neutrinos=Nncdm, standard_neutrino_neff=3.046, 
                       mnu=np.sum(mnus))   ##
    camb_results = camb.get_background(pars)
    return camb_results
thickness = 50 # Mpc/h

ic    = 0
fdir  = "/home/cossim/yuyu22/chenzhao/csst/simulation/c%04d/output/"%(ic)
fpart = fdir + "snapdir_000_down64/snapshot_000_down64.0.hdf5"
camb_results = Getcamb(fpart)
### request z [zmin, zmax]
zmin = 0
zmax = 1.0
chi_min = camb_results.comoving_radial_distance(zmin) * camb_results.Params.h
chi_max = camb_results.comoving_radial_distance(zmax) * camb_results.Params.h
print("requst z in          [%.2f, %.2f]."%(zmin,zmax) )
print("comoving distance in [%.0f, %.0f]"%(chi_min,chi_max))
ishell_min = int(np.floor(chi_min/thickness))
ishell_max = int(np.floor(chi_max/thickness))
print("shell index in       [%d, %d]"%(ishell_min,ishell_max))

hdir  = fdir + "fullskyhalo-rockstar/"
fnames  = glob(hdir + "mapsdir*")
imm_all = [int(fname.split("_")[-1]) for fname in fnames]
imm_max = np.max(imm_all)
print("Maximum index of shells: %d"%(imm_max))
readkeys = ['Rs', 'Rvir', 'Vrms', 'VX', 'VY', 'VZ']
readkeys_h = readkeys.copy()
readkeys_s = readkeys.copy()
readkeys_h.append('GroupPos')
readkeys_s.append('SubhaloPos')
# for ishell in range(ishell_min, ishell_max+1):
for ishell in range(1):
    imm   = imm_max - ishell  # for chi ~ [i*thickness, (i+1)*thickness]
    fname = hdir + "mapsdir_%03d/halo_sub_%03d.0.hdf5"%(imm, imm)
    with h5py.File(fname, "r") as ff:
        snapnum = ff['Header'].attrs['SnapNum']
    print("Corrsponding snap_%03d"%(snapnum))
    halos = IO_Halo_multifile(fname, readkeys=readkeys_h, subhalo=False, n_processes=1)
    print("Check Halo")
    for ii in range(3):
        print(halos['GroupPos'][:,ii].min(), halos['GroupPos'][:,ii].max())
    subhalos = IO_Halo_multifile(fname, readkeys=readkeys_s, subhalo=True, n_processes=1)
    print("Check Subhalo")
    for ii in range(3):
        print(subhalos['SubhaloPos'][:,ii].min(), subhalos['SubhaloPos'][:,ii].max())
    ### check wether in lightcone_00 or lightcone_01
    ind00  = is_in_cone(halos['GroupPos'], lc_dict['00'])
    ind01  = is_in_cone(halos['GroupPos'], lc_dict['01'])
    indall = ind00 + ind01
    halos  = halos[indall]
    print("Check Halo")
    print(halos.shape)
    for ii in range(3):
        print(halos['GroupPos'][:,ii].min(), halos['GroupPos'][:,ii].max())
    ind00  = is_in_cone(subhalos['SubhaloPos'], lc_dict['00'])
    ind01  = is_in_cone(subhalos['SubhaloPos'], lc_dict['01'])
    indall = ind00 + ind01
    subhalos  = subhalos[indall]
    print("Check Subhalo")
    print(subhalos.shape)
    for ii in range(3):
        print(subhalos['SubhaloPos'][:,ii].min(), subhalos['SubhaloPos'][:,ii].max())
    ### add concentration
    halos = rfn.append_fields(halos,                   # 原数组
                              names='cvir',            # 新字段名
                              data=halos['Rvir']/halos['Rs'],  # 新字段值
                              dtypes='<f8',            # 新字段类型
                              usemask=False            # 非掩码数组，必加
                            )
    print("concentration of halos: min=%.2f, max=%.2f"%(halos['cvir'].min(), halos['cvir'].max()))

requst z in          [0.00, 1.00].
comoving distance in [0, 2297]
shell index in       [0, 45]
Maximum index of shells: 89
Corrsponding snap_011
Reading 64 files with 1 processes
Concatenating data from 9 files...
Total 38186 halos read
IO_Halo_multifile CPU  Time: 0.05843450899999425
IO_Halo_multifile Wall Time: 0.08356857299804688
Check Halo
-49.85626000000002 49.34273
-49.50427000000002 49.93416
-49.899779999999964 49.83842
Reading 64 files with 1 processes
Concatenating data from 8 files...
Total 5851 halos read
IO_Halo_multifile CPU  Time: 0.034377519000003076
IO_Halo_multifile Wall Time: 0.034377336502075195
Check Subhalo
-48.454830000000015 48.1164
-48.983460000000036 49.75857
-48.836490000000026 49.73729
Check Halo
(3542,)
-2.252200000000016 49.34273
-2.507259999999974 49.6707
-1.4912699999999859 32.85165
Check Subhalo
(494,)
-1.0137300000000096 45.96037
-2.082210000000032 49.75857
-1.4373199999999997 30.68252
concentration of halos: min=1.03, max=48.68


In [35]:
print(halos.dtype)
print(subhalos.dtype)

[('Rs', '<f8'), ('Rvir', '<f8'), ('Vrms', '<f8'), ('VX', '<f8'), ('VY', '<f8'), ('VZ', '<f8'), ('GroupPos', '<f8', (3,)), ('cvir', '<f8')]
[('Rs', '<f8'), ('Rvir', '<f8'), ('Vrms', '<f8'), ('VX', '<f8'), ('VY', '<f8'), ('VZ', '<f8'), ('SubhaloPos', '<f8', (3,))]


# use healpy mask and search more property

In [9]:
import sys
sys.path.append("/home/chenzhao/csst/simulation/scripts/")
from LightConeInfo import lc_dict, is_in_cone
sys.path.append("/home/chenzhao/worksoftware/csst-simulations-read/KunRead/")
from io_utils import IO_Halo_single, IO_Halo_multifile, get_available_fields
from glob import glob
import numpy as np
import camb
import h5py

def Getcamb(hdf5file):
    import h5py
    header = {}
    with h5py.File(hdf5file, 'r') as ff:
        for ikey in ff['Parameters'].attrs.keys():
            header[ikey] = ff['Parameters'].attrs[ikey]
    pars = camb.CAMBparams()
    param_in = {}
    ddir = '/home/chenzhao/csst/cosmologies'
    h0 = header['HubbleParam']
    Omegab = header['OmegaBaryon']
    Omegac = header['Omega0'] - Omegab
    omch2  = Omegac*h0*h0
    ombh2  = Omegab*h0*h0
    mnus   = [header['m_nu1'], header['m_nu2'], header['m_nu3']]
    Nncdm  = len(np.where(mnus != 0)[0])
    pars.set_cosmology(H0=h0*100, omch2=omch2, ombh2=ombh2, 
                       num_massive_neutrinos=Nncdm, standard_neutrino_neff=3.046, 
                       mnu=np.sum(mnus))   ##
    camb_results = camb.get_background(pars)
    return camb_results
thickness = 50 # Mpc/h

ic    = 0
fdir  = "/home/cossim/yuyu22/chenzhao/csst/simulation/c%04d/output/"%(ic)
fpart = fdir + "snapdir_000_down64/snapshot_000_down64.0.hdf5"
camb_results = Getcamb(fpart)
### request z [zmin, zmax]
zmin = 0
zmax = 1.0
chi_min = camb_results.comoving_radial_distance(zmin) * camb_results.Params.h
chi_max = camb_results.comoving_radial_distance(zmax) * camb_results.Params.h
print("requst z in          [%.2f, %.2f]."%(zmin,zmax) )
print("comoving distance in [%.0f, %.0f]"%(chi_min,chi_max))
ishell_min = int(np.floor(chi_min/thickness))
ishell_max = int(np.floor(chi_max/thickness))
print("shell index in       [%d, %d]"%(ishell_min,ishell_max))

hdir  = fdir + "fullskyhalo-rockstar/"
fnames  = glob(hdir + "mapsdir*")
imm_all = [int(fname.split("_")[-1]) for fname in fnames]
imm_max = np.max(imm_all)
print("Maximum index of shells: %d"%(imm_max))
readkeys = ['ID', 'Rs', 'Rvir', 'Vrms', 'VX', 'VY', 'VZ']
snapnum_read = None
# for ishell in range(ishell_min, ishell_max+1):
for ishell in range(1):
    imm   = imm_max - ishell  # for chi ~ [i*thickness, (i+1)*thickness]
    fname = hdir + "mapsdir_%03d/halo_sub_%03d.0.hdf5"%(imm, imm)
    with h5py.File(fname, "r") as ff:
        snapnum = ff['Header'].attrs['SnapNum']
    print("Corrsponding snap_%03d"%(snapnum))
    halos = IO_Halo_multifile(fname, readkeys=['GroupPos', 'ID'], subhalo=False, n_processes=1)
    print("Check Halo")
    for ii in range(3):
        print(halos['GroupPos'][:,ii].min(), halos['GroupPos'][:,ii].max())
    subhalos = IO_Halo_multifile(fname, readkeys=['SubhaloPos', 'ID'], subhalo=True, n_processes=1)
    print("Check Subhalo")
    for ii in range(3):
        print(subhalos['SubhaloPos'][:,ii].min(), subhalos['SubhaloPos'][:,ii].max())
    ### check wether in lightcone_00 or lightcone_01
    ind00  = is_in_cone(halos['GroupPos'], lc_dict['00'])
    ind01  = is_in_cone(halos['GroupPos'], lc_dict['01'])
    indall = ind00 + ind01
    halos  = halos[indall]
    print("Check Halo")
    print(halos.shape)
    for ii in range(3):
        print(halos['GroupPos'][:,ii].min(), halos['GroupPos'][:,ii].max())
    ind00  = is_in_cone(subhalos['SubhaloPos'], lc_dict['00'])
    ind01  = is_in_cone(subhalos['SubhaloPos'], lc_dict['01'])
    indall = ind00 + ind01
    subhalos  = subhalos[indall]
    print("Check Subhalo")
    print(subhalos.shape)
    for ii in range(3):
        print(subhalos['SubhaloPos'][:,ii].min(), subhalos['SubhaloPos'][:,ii].max())
    if snapnum_read != snapnum:
        print("Load snap halo catalog for more halo properties ...")
        fhalo = fdir + "groups_%03d_rockstar/rockstar_tab_%03d.0.hdf5"%(snapnum, snapnum)
        ## for halo
        snaphalos = IO_Halo_multifile(fhalo, readkeys=readkeys, subhalo=False, n_processes=1)
        snapnum_read = snapnum
    # snapsubhalos = IO_Halo_multifile(fhalo, readkeys=readkeys, subhalo=True)

requst z in          [0.00, 1.00].
comoving distance in [0, 2297]
shell index in       [0, 45]
Maximum index of shells: 89
Corrsponding snap_011
Reading 64 files with 1 processes
Concatenating data from 9 files...
Total 38186 halos read
IO_Halo_multifile CPU  Time: 0.03159311499999973
IO_Halo_multifile Wall Time: 0.043997764587402344
Check Halo
-49.85626000000002 49.34273
-49.50427000000002 49.93416
-49.899779999999964 49.83842
Reading 64 files with 1 processes
Concatenating data from 8 files...
Total 5851 halos read
IO_Halo_multifile CPU  Time: 0.022024521999995272
IO_Halo_multifile Wall Time: 0.02147984504699707
Check Subhalo
-48.454830000000015 48.1164
-48.983460000000036 49.75857
-48.836490000000026 49.73729
Check Halo
(3542,)
-2.252200000000016 49.34273
-2.507259999999974 49.6707
-1.4912699999999859 32.85165
Check Subhalo
(494,)
-1.0137300000000096 45.96037
-2.082210000000032 49.75857
-1.4373199999999997 30.68252
Load snap halo catalog for more halo properties ...
Reading 64 files

In [10]:
print(halos.dtype)
print(subhalos.dtype)
print(snaphalos.dtype)

[('GroupPos', '<f8', (3,)), ('ID', '<f8')]
[('SubhaloPos', '<f8', (3,)), ('ID', '<f8')]
[('ID', '<f8'), ('Rs', '<f8'), ('Rvir', '<f8'), ('Vrms', '<f8'), ('VX', '<f8'), ('VY', '<f8'), ('VZ', '<f8')]


In [21]:
def MatchID(halos, snaphalos):
    '''
    match ID between fullsky halo and snapshot halo
    
    Args:
    -----
    halos: readed halos or subhalos, must contain 'ID'
    snaphalos: readed halos or subhalos from snapshot, must contain 'ID'
    Return:
    -------
    halos_new
    '''
    # ================================
    # 1. 构建 ID → 索引 映射（核心提速步骤）
    # ================================
    # Get ID in snaphalos
    snap_ids = snaphalos['ID']
    # 构建字典：ID值 → 对应行索引（O(1)查找，最快）
    id_to_idx = {id_val: idx for idx, id_val in enumerate(snap_ids)}
    # ========================================
    # 2. 批量匹配halos的ID，获取对应snaphalos索引
    # ========================================
    halo_ids = halos['ID']
    # 批量查找索引（不存在的ID赋值为-1）
    matched_indices = np.array([id_to_idx.get(id_val, -1) for id_val in halo_ids])
    # 过滤存在的ID
    valid_mask = matched_indices != -1
    # =====================================
    # 3. 为halos数组新增需要的字段（定义新dtype）
    # =====================================
    # 原始dtype + 新增的snaphalos属性
    new_dtype = [(ikey, snaphalos[ikey].dtype) for ikey in snaphalos.dtype.names]
    new_dtype.append(("GroupPos", halos['GroupPos'].dtype, (3,)))
    # print(new_dtype)
    # 创建新数组，拷贝原有数据
    halos_new = np.empty_like(halos, dtype=new_dtype)
    for name in halos.dtype.names:
        halos_new[name] = halos[name]
    # ========================
    # 4. 批量赋值（最高效，无循环）
    # ========================
    # 仅对匹配成功的行赋值
    valid_idx = matched_indices[valid_mask]
    attrs     = list(snaphalos.dtype.names)
    attrs.remove("ID")
    print("新增字段: ", attrs)
    for attr in attrs:
        halos_new[attr][valid_mask] = snaphalos[attr][valid_idx]
    return halos_new

halos_new = MatchID(halos, snaphalos)
# =================================
# 最终结果：halos_new 就是匹配完成的数组
# =================================
print("halos新形状："  , halos_new.shape)
print("halos新dtype：", halos_new.dtype)

[('ID', dtype('<f8')), ('Rs', dtype('<f8')), ('Rvir', dtype('<f8')), ('Vrms', dtype('<f8')), ('VX', dtype('<f8')), ('VY', dtype('<f8')), ('VZ', dtype('<f8')), ('GroupPos', dtype('<f8'), (3,))]
新增字段:  ['Rs', 'Rvir', 'Vrms', 'VX', 'VY', 'VZ']
halos新形状： (3542,)
halos新dtype： [('ID', '<f8'), ('Rs', '<f8'), ('Rvir', '<f8'), ('Vrms', '<f8'), ('VX', '<f8'), ('VY', '<f8'), ('VZ', '<f8'), ('GroupPos', '<f8', (3,))]
